# Thesis snapshot: historical audit and corrected timing

Requires the private `data/local/df3.csv` recovered from the original project. If absent, the notebook explains how to proceed and finishes without market results. Corrected results are new analyses and must not be attributed to the submitted report.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from commodity_qr.analysis import *


In [ ]:
path = ROOT / "data/local/df3.csv"
snapshot = load_snapshot(path) if path.exists() else None
if snapshot is None:
    print("Local snapshot absent. See data/README.md; run the synthetic demo instead.")
else:
    print(f"Snapshot: {len(snapshot)} rows; {snapshot.index.min().date()} to {snapshot.index.max().date()}")

## Historical audit

Retains the original future-valued regressor to investigate the historical tables. The full notebook dropped 19 observations for a rolling volatility calculation before regression. These historical estimates must not be described as a valid autoregression.

In [ ]:
runs = {}
if snapshot is not None:
    for mode in ("historical", "corrected"):
        frame = prepare_snapshot(snapshot, mode=mode)
        for spec in ("full", "report_reduced", "notebook_reduced"):
            key = f"{mode}_{spec}"
            runs[key] = save_outputs(frame, ROOT / "results" / key, specification=spec, label=key)
    display(runs["historical_full"][1])

## Corrected explanatory analysis

Returns are reassigned to the day they end; the lag is then built from past observations. VIX is contemporaneous. Regime boundaries follow the report. Both full and reduced specifications are exported explicitly; the original reduced notebook and report use different interaction sets.

In [ ]:
if snapshot is not None:
    from IPython.display import Image, display
    display(Image(filename=str(ROOT / "results/corrected_full/quantile_summary.png")))
    display(runs["corrected_full"][1])

## Before interpreting

Read `docs/reconciliation.md` and `docs/validation.md`. The snapshot lacks raw index/FX levels, so FX conversion and upstream return construction cannot be independently verified from this file. Overlapping sessions and synchronous VIX timing prevent a tradable forecast interpretation. The invalid independent-estimates Wald comparison from the originals is deliberately omitted.